In [ ]:
import os
os.chdir("/home/ubuntu/work/dahyeon/backend")
os.getcwd()

'/home/ubuntu/work/dahyeon/backend'

In [42]:
import asyncio
import json
import uuid
import time
import random
import requests
import nest_asyncio
from pathlib import Path
from collections import defaultdict
from tqdm.auto import tqdm
from app.modules.RAG.retriever import (
    full_hybrid
)
import sys
from app.modules.RAG.anchor_book_pipeline import run_anchor_pipeline
from app.core.config import settings
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
nest_asyncio.apply()

/home/ubuntu/anaconda3/envs/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/ubuntu/anaconda3/envs/venv/lib/python3.11/site-packages/elasticsearch/_sync/client/__init__.py:404: SecurityWarning: Connecting to 'https://localhost:9200' using TLS with verify_certs=False is insecure
  _transport = transport_class(


In [3]:
REPO_ROOT     = Path("/home/ubuntu/work/dahyeon")
DATASET_DIR   = REPO_ROOT / "evaluation" / "dataset"
SCENARIO_PATH = DATASET_DIR / "scenario_data_v2.json"

with open(SCENARIO_PATH, encoding="utf-8") as f:
    scenarios = json.load(f)

def extract_rag_query(scenario: dict) -> dict:
    """마지막 turn의 rag_query를 꺼내고 query_id(=scenario_id)를 추가한다."""
    rag_query = scenario["turns"][-1]["rag_query"].copy()
    rag_query["query_id"] = scenario["scenario_id"]
    return rag_query

In [4]:
def simplify_item(item, source_name):
    return {
        "isbn":         item.get("isbn"),
        "title":        item.get("title"),
        "author":       item.get("author"),
        "publisher":    item.get("publisher"),
        "publish_date": item.get("publish_date"),
        "page":         item.get("page"),
        "large_cate":   item.get("large_cate"),
        "mid_cate":     item.get("mid_cate"),
        "small_cate":   item.get("small_cate"),
        "book_intro":   item.get("book_intro"),
        "book_index":   item.get("book_index"),
        "review_count": item.get("review_count"),
        "review_text":  item.get("review_text"),
    }


In [5]:
# ============================================================
# Strategy0 Hybrid BM25/Dense 가중치 실험
# - index: books_review_full
# - retrieval: full_hybrid
# - embedding model: KURE
# - bm25_weight / dense_weight 조절
# ============================================================

from pathlib import Path
import json
import pandas as pd
from tqdm import tqdm

INDEX_NAME = "books_review_full_100000"

EMBEDDING_MODEL = "kure"
EMBEDDING_FIELD = "embedding_kure"

TOP_K = 20
BM25_CANDIDATE_SIZE = 100
DENSE_CANDIDATE_SIZE = 100
NUM_CANDIDATES = 300

WEIGHT_CONFIGS = [
    {"bm25_weight": 0.3, "dense_weight": 0.7}
]

OUTPUT_PATH = Path("/home/ubuntu/work/dahyeon/evaluation/dataset/final_retriever_eval.jsonl")

all_candidates = []

for scenario in tqdm(scenarios, desc="strategy0 hybrid weight experiment"):

    sample_result = extract_rag_query(scenario)

    if sample_result.get("anchor"):
        sample_result = run_anchor_pipeline(sample_result)

    query_id = sample_result["query_id"]

    for weight_id, weights in enumerate(WEIGHT_CONFIGS, start=1):

        bm25_weight = weights["bm25_weight"]
        dense_weight = weights["dense_weight"]

        re_hybrid = full_hybrid(
            result=sample_result,
            index_name=INDEX_NAME,
            size=TOP_K,
            bm25_candidate_size=BM25_CANDIDATE_SIZE,
            dense_candidate_size=DENSE_CANDIDATE_SIZE,
            num_candidates=NUM_CANDIDATES,
            require_both=False,
            overlap_bonus=0.0,
            rrf_k=60,
            bm25_weight=bm25_weight,
            dense_weight=dense_weight,
            embedding_field=EMBEDDING_FIELD,
            embedding_model=EMBEDDING_MODEL
        )

        for rank, book in enumerate(re_hybrid, start=1):
            all_candidates.append({
                "query_id": query_id,

                "strategy": "strategy0",
                "retrieval_type": "full_hybrid",
                "index_name": INDEX_NAME,

                "weight_id": weight_id,
                "bm25_weight": bm25_weight,
                "dense_weight": dense_weight,

                "rank": rank,
                "score": book.get("score"),

                "bm25_rank": book.get("bm25_rank"),
                "dense_rank": book.get("dense_rank"),
                "bm25_raw_score": book.get("bm25_raw_score"),
                "dense_raw_score": book.get("dense_raw_score"),
                "bm25_rrf_score": book.get("bm25_rrf_score"),
                "dense_rrf_score": book.get("dense_rrf_score"),
                "has_bm25": book.get("has_bm25"),
                "has_dense": book.get("has_dense"),

                "isbn": str(book.get("isbn")),
                "title": book.get("title"),
                "author": book.get("author"),
                "publisher": book.get("publisher"),
                "publish_date": book.get("publish_date"),
                "page": book.get("page"),

                "large_cate": book.get("large_cate") or book.get("large"),
                "mid_cate": book.get("mid_cate"),
                "small_cate": book.get("small_cate"),

                "book_intro": book.get("book_intro"),
                "book_index": book.get("book_index"),
                "book_index_text": book.get("book_index_text"),
                "review": book.get("review"),
                "review_text": book.get("review_text"),
            })

print(f"\n전체 후보 도서: {len(all_candidates):,}")

with OUTPUT_PATH.open("w", encoding="utf-8") as f:
    for item in all_candidates:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"저장 완료: {OUTPUT_PATH}")

strategy0_weight_df = pd.DataFrame(all_candidates)
strategy0_weight_df.head()

strategy0 hybrid weight experiment: 100%|██████████| 42/42 [02:29<00:00,  3.55s/it]


전체 후보 도서: 840
저장 완료: /home/ubuntu/work/dahyeon/evaluation/dataset/final_retriever_eval.jsonl


,query_id,strategy,retrieval_type,index_name,weight_id,bm25_weight,dense_weight,rank,score,bm25_rank,...,publish_date,page,large_cate,mid_cate,small_cate,book_intro,book_index,book_index_text,review,review_text
0,S01,strategy0,full_hybrid,books_review_full_100000,1,0.3,0.7,1,0.014924,27.0,...,2025-03-10,232,"[시/에세이, 인문]",[나라별 에세이],[한국에세이],"“인간존재와 삶의 의미는 무엇인지, 왜 사는지, 무엇을 위해 어떻게 살 것인지, 늙...","[잠언 편, [ ], 삶의 본질에 대한 질문과 대답, 영적 세계와 앎에 대한 사유,...",잠언 편 [ ] 삶의 본질에 대한 질문과 대답 영적 세계와 앎에 대한 사유 삶의 방...,{'reader_reaction': '속이 뻥 뚫히며 모든 근심을 다 내려놓게 만드...,속이 뻥 뚫히며 모든 근심을 다 내려놓게 만드는 책입니다. 삶의 본질을 알고자 하거...
1,S01,strategy0,full_hybrid,books_review_full_100000,1,0.3,0.7,2,0.014815,21.0,...,2024-09-20,284,[시/에세이],"[나라별 에세이, 철학]","[철학에세이, 한국에세이]",세상을 건너다 종종 길을 잃는 그대에게\n노교수가 전하는 성찰의 메시지 68\n인간...,"[왜 살아야 하는지를 아는 사람, 남몰래 흐르는 눈물, 아무것도 하지 않기, 너는 ...",왜 살아야 하는지를 아는 사람 남몰래 흐르는 눈물 아무것도 하지 않기 너는 나의 자부심,"{'reader_reaction': '저자의 통찰력에 감탄했으며, 관계론적 존재론에...","저자의 통찰력에 감탄했으며, 관계론적 존재론에 대한 흥미로운 시각을 제시합니다. 저..."
2,S01,strategy0,full_hybrid,books_review_full_100000,1,0.3,0.7,3,0.013756,12.0,...,2025-03-19,168,[시/에세이],[한국시],[현대시],"《경계없는 놀이터》는 수필 같고, 소설 같고, 자기계발 같으며, 철학적 메시지를 품...","[건강한 욕망, 나, 자신감, 내가 내 삶에 자신감이 있으려면, 소유, 임시적 착각...","건강한 욕망 나, 자신감 내가 내 삶에 자신감이 있으려면 소유 임시적 착각의 신뢰",{'reader_reaction': '읽다가 멈추고 생각하게 되는 문장이 많아 천천...,읽다가 멈추고 생각하게 되는 문장이 많아 천천히 읽게 됩니다. 짧은 문장과 단락에도...
3,S01,strategy0,full_hybrid,books_review_full_100000,1,0.3,0.7,4,0.013614,23.0,...,2025-01-25,280,[시/에세이],"[나라별 에세이, 테마에세이]","[포토에세이, 한국에세이]","스토리 천국, 세상의 울림에 귀 기울이는 어느 사회과학자의 사진 일기\n이 책은 2...","[길은 사유를 낳고, 저녁이 깃든 포구, 어항의 아침, 자연에서 배운다, 산 공화국...",길은 사유를 낳고 저녁이 깃든 포구 어항의 아침 자연에서 배운다 산 공화국 꽃과 나...,None,
4,S01,strategy0,full_hybrid,books_review_full_100000,1,0.3,0.7,5,0.013169,65.0,...,2024-12-09,183,[시/에세이],[],[],"이 책은 일상의 작은 순간부터 인생을 바꾸는 중요한 결정까지, 삶의 다양한 면을 깊...","[삶이란 무엇인가, 하루의 시작과 끝]",삶이란 무엇인가 하루의 시작과 끝,None,


In [7]:
from pathlib import Path
import sys
import json
import pandas as pd

# =====================================================
# 1. 파일 경로 / 설정
# =====================================================

GOLDSET_PATH = Path("/home/ubuntu/work/dahyeon/evaluation/dataset/goldset_final_v2.json")
RETRIEVAL_RESULT_PATH = Path("/home/ubuntu/work/dahyeon/evaluation/dataset/final_retriever_eval.jsonl")

REPO_ROOT = Path("/home/ubuntu/work/dahyeon")
sys.path.insert(0, str(REPO_ROOT / "evaluation" / "metrics"))

from metrics import hit_at_k, recall_at_k, mrr_at_k, ndcg_at_k

K = 20
RELEVANCE_THRESHOLD = 2

# =====================================================
# 2. goldset 로드
# =====================================================

with GOLDSET_PATH.open("r", encoding="utf-8") as f:
    goldset = json.load(f)

gold_df = pd.DataFrame(goldset)

relevant_by_query = {}

for qid, g in gold_df.groupby("query_id"):
    relevant_isbns = (
        g[g["final_grade"] >= RELEVANCE_THRESHOLD]["isbn"]
        .dropna()
        .astype(str)
        .tolist()
    )
    relevant_by_query[qid] = set(relevant_isbns)

print("전체 query 수:", len(relevant_by_query))
print("relevant 있는 query 수:", sum(1 for v in relevant_by_query.values() if v))

# =====================================================
# 3. 최종 Hybrid retrieval 결과 로드
# =====================================================

rows = []

with RETRIEVAL_RESULT_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            rows.append(json.loads(line))

retrieval_df = pd.DataFrame(rows)

print("retrieval 결과 수:", len(retrieval_df))
print("query 수:", retrieval_df["query_id"].nunique())

display(retrieval_df.head())

# =====================================================
# 4. query별 ranked isbn 구성
# =====================================================

ranked_by_query = {}

for qid, g in retrieval_df.groupby("query_id"):
    ranked = (
        g.sort_values("rank")["isbn"]
        .dropna()
        .astype(str)
        .tolist()
    )
    ranked_by_query[qid] = ranked

# =====================================================
# 5. 최종 Hybrid 평가
# =====================================================

metric_rows = []

for qid, rel in relevant_by_query.items():

    if not rel:
        continue

    ranked = ranked_by_query.get(qid, [])

    if not ranked:
        metric_rows.append({
            "query_id": qid,
            "k": K,
            "hit": 0,
            "recall": 0.0,
            "mrr": 0.0,
            "ndcg": 0.0,
            "goldset_count": len(rel),
            "retrieved_relevant_count": 0,
            "matched_isbns": [],
            "retrieved_count": 0,
        })
        continue

    topk = ranked[:K]
    hit_isbns = set(topk) & rel

    metric_rows.append({
        "query_id": qid,
        "k": K,
        "hit": hit_at_k(rel, ranked, K),
        "recall": recall_at_k(rel, ranked, K),
        "mrr": mrr_at_k(rel, ranked, K),
        "ndcg": ndcg_at_k(rel, ranked, K),
        "goldset_count": len(rel),
        "retrieved_relevant_count": len(hit_isbns),
        "matched_isbns": list(hit_isbns),
        "retrieved_count": len(topk),
    })

final_eval_df = pd.DataFrame(metric_rows)

print("평가 query 수:", len(final_eval_df))
display(final_eval_df.head())

# =====================================================
# 6. 최종 요약
# =====================================================

summary = {
    "strategy": "strategy0",
    "retrieval_type": "full_hybrid",
    "index_name": retrieval_df["index_name"].iloc[0] if "index_name" in retrieval_df.columns else None,
    "bm25_weight": retrieval_df["bm25_weight"].iloc[0] if "bm25_weight" in retrieval_df.columns else None,
    "dense_weight": retrieval_df["dense_weight"].iloc[0] if "dense_weight" in retrieval_df.columns else None,
    "k": K,
    "mean_hit": final_eval_df["hit"].mean(),
    "mean_recall": final_eval_df["recall"].mean(),
    "mean_mrr": final_eval_df["mrr"].mean(),
    "mean_ndcg": final_eval_df["ndcg"].mean(),
    "total_goldset_count": final_eval_df["goldset_count"].sum(),
    "total_retrieved_relevant_count": final_eval_df["retrieved_relevant_count"].sum(),
}

summary_df = pd.DataFrame([summary])

display(summary_df)

print("=" * 60)
print("최종 Hybrid 평가 결과")
print("=" * 60)
print("strategy:", summary["strategy"])
print("retrieval_type:", summary["retrieval_type"])
print("index_name:", summary["index_name"])
print("bm25_weight:", summary["bm25_weight"])
print("dense_weight:", summary["dense_weight"])
print(f"Hit@{K}:", round(summary["mean_hit"], 4))
print(f"Recall@{K}:", round(summary["mean_recall"], 4))
print(f"MRR@{K}:", round(summary["mean_mrr"], 4))
print(f"NDCG@{K}:", round(summary["mean_ndcg"], 4))
print("전체 relevant 수:", summary["total_goldset_count"])
print("검색된 relevant 수:", summary["total_retrieved_relevant_count"])
print("=" * 60)

전체 query 수: 39
relevant 있는 query 수: 39
retrieval 결과 수: 840
query 수: 42


,query_id,strategy,retrieval_type,index_name,weight_id,bm25_weight,dense_weight,rank,score,bm25_rank,...,publish_date,page,large_cate,mid_cate,small_cate,book_intro,book_index,book_index_text,review,review_text
0,S01,strategy0,full_hybrid,books_review_full_100000,1,0.3,0.7,1,0.014924,27.0,...,2025-03-10,232,"[시/에세이, 인문]",[나라별 에세이],[한국에세이],"“인간존재와 삶의 의미는 무엇인지, 왜 사는지, 무엇을 위해 어떻게 살 것인지, 늙...","[잠언 편, [ ], 삶의 본질에 대한 질문과 대답, 영적 세계와 앎에 대한 사유,...",잠언 편 [ ] 삶의 본질에 대한 질문과 대답 영적 세계와 앎에 대한 사유 삶의 방...,{'reader_reaction': '속이 뻥 뚫히며 모든 근심을 다 내려놓게 만드...,속이 뻥 뚫히며 모든 근심을 다 내려놓게 만드는 책입니다. 삶의 본질을 알고자 하거...
1,S01,strategy0,full_hybrid,books_review_full_100000,1,0.3,0.7,2,0.014815,21.0,...,2024-09-20,284,[시/에세이],"[나라별 에세이, 철학]","[철학에세이, 한국에세이]",세상을 건너다 종종 길을 잃는 그대에게\n노교수가 전하는 성찰의 메시지 68\n인간...,"[왜 살아야 하는지를 아는 사람, 남몰래 흐르는 눈물, 아무것도 하지 않기, 너는 ...",왜 살아야 하는지를 아는 사람 남몰래 흐르는 눈물 아무것도 하지 않기 너는 나의 자부심,"{'reader_reaction': '저자의 통찰력에 감탄했으며, 관계론적 존재론에...","저자의 통찰력에 감탄했으며, 관계론적 존재론에 대한 흥미로운 시각을 제시합니다. 저..."
2,S01,strategy0,full_hybrid,books_review_full_100000,1,0.3,0.7,3,0.013756,12.0,...,2025-03-19,168,[시/에세이],[한국시],[현대시],"《경계없는 놀이터》는 수필 같고, 소설 같고, 자기계발 같으며, 철학적 메시지를 품...","[건강한 욕망, 나, 자신감, 내가 내 삶에 자신감이 있으려면, 소유, 임시적 착각...","건강한 욕망 나, 자신감 내가 내 삶에 자신감이 있으려면 소유 임시적 착각의 신뢰",{'reader_reaction': '읽다가 멈추고 생각하게 되는 문장이 많아 천천...,읽다가 멈추고 생각하게 되는 문장이 많아 천천히 읽게 됩니다. 짧은 문장과 단락에도...
3,S01,strategy0,full_hybrid,books_review_full_100000,1,0.3,0.7,4,0.013614,23.0,...,2025-01-25,280,[시/에세이],"[나라별 에세이, 테마에세이]","[포토에세이, 한국에세이]","스토리 천국, 세상의 울림에 귀 기울이는 어느 사회과학자의 사진 일기\n이 책은 2...","[길은 사유를 낳고, 저녁이 깃든 포구, 어항의 아침, 자연에서 배운다, 산 공화국...",길은 사유를 낳고 저녁이 깃든 포구 어항의 아침 자연에서 배운다 산 공화국 꽃과 나...,None,
4,S01,strategy0,full_hybrid,books_review_full_100000,1,0.3,0.7,5,0.013169,65.0,...,2024-12-09,183,[시/에세이],[],[],"이 책은 일상의 작은 순간부터 인생을 바꾸는 중요한 결정까지, 삶의 다양한 면을 깊...","[삶이란 무엇인가, 하루의 시작과 끝]",삶이란 무엇인가 하루의 시작과 끝,None,


평가 query 수: 39


,query_id,k,hit,recall,mrr,ndcg,goldset_count,retrieved_relevant_count,matched_isbns,retrieved_count
0,S01,20,1,0.444444,1.0,0.535840,9,4,"[9791196539757, 9791157833733, 9791193063798, ...",20
1,S03,20,1,0.393939,1.0,0.702552,33,13,"[9791186889275, 9791186900895, 9791191926859, ...",20
2,S04,20,1,0.551724,1.0,0.859936,29,16,"[9791168321175, 9788990989604, 9788989797111, ...",20
3,S05,20,1,0.407407,0.5,0.537781,27,11,"[9791156029182, 9791192376585, 9788997294411, ...",20
4,S06,20,1,0.303571,1.0,0.831273,56,17,"[9791194006435, 9791192259116, 9788974678401, ...",20


,strategy,retrieval_type,index_name,bm25_weight,dense_weight,k,mean_hit,mean_recall,mean_mrr,mean_ndcg,total_goldset_count,total_retrieved_relevant_count
0,strategy0,full_hybrid,books_review_full_100000,0.3,0.7,20,1.0,0.358607,0.909188,0.614617,1258,418


최종 Hybrid 평가 결과
strategy: strategy0
retrieval_type: full_hybrid
index_name: books_review_full_100000
bm25_weight: 0.3
dense_weight: 0.7
Hit@20: 1.0
Recall@20: 0.3586
MRR@20: 0.9092
NDCG@20: 0.6146
전체 relevant 수: 1258
검색된 relevant 수: 418


In [8]:
from pathlib import Path
import json

# =====================================================
# 파일 경로
# =====================================================

JSONL_PATH = Path("/home/ubuntu/work/dahyeon/evaluation/dataset/final_retriever_eval.jsonl")
OUTPUT_PATH = Path("/home/ubuntu/work/dahyeon/evaluation/dataset/final_retriever_eval.json")

# =====================================================
# jsonl -> json 변환
# =====================================================

rows = []

with JSONL_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()

        if not line:
            continue

        rows.append(json.loads(line))

# =====================================================
# 저장
# =====================================================

with OUTPUT_PATH.open("w", encoding="utf-8") as f:
    json.dump(
        rows,
        f,
        ensure_ascii=False,
        indent=2
    )

print("=" * 50)
print("변환 완료")
print("jsonl row 수:", len(rows))
print("저장 위치:", OUTPUT_PATH)
print("=" * 50)

변환 완료
jsonl row 수: 840
저장 위치: /home/ubuntu/work/dahyeon/evaluation/dataset/final_retriever_eval.json


# 정성적 평가 LLM-judge

In [9]:
import os
os.chdir("/home/ubuntu/work/dahyeon/backend")
os.getcwd()

'/home/ubuntu/work/dahyeon/backend'

In [10]:
import asyncio
import json
import uuid
import time
import random
import requests
import nest_asyncio
from pathlib import Path
from collections import defaultdict
from tqdm.auto import tqdm
from app.core.config import settings
import pandas as pd

import warnings
warnings.filterwarnings('ignore')

nest_asyncio.apply()

In [45]:
import json
import uuid
import time
import random
import asyncio
import requests
from pathlib import Path
from tqdm import tqdm

# =====================================================
# 0. 설정
# =====================================================

URL           = "https://clovastudio.stream.ntruss.com/v3/chat-completions/HCX-007"
CLOVA_API_KEY = settings.CLOVA_API_KEY

GOLDSET_PATH = Path("/home/ubuntu/work/dahyeon/evaluation/dataset/goldset_final_v2.json")
SCENARIO_PATH = Path("/home/ubuntu/work/dahyeon/evaluation/dataset/scenario_data_v2.json")
RETRIEVAL_JSON_PATH = Path("/home/ubuntu/work/dahyeon/evaluation/dataset/final_retriever_eval.json")
JSONL_PATH = Path("/home/ubuntu/work/dahyeon/evaluation/dataset/final_retriever_llm_judge.jsonl")
FINAL_JSON_PATH = Path("/home/ubuntu/work/dahyeon/evaluation/dataset/final_retriever_llm_judge.json")

scenario_map = {}
hard_negative_map = {}

TOP_K = 20
CONCURRENCY = 3
MAX_RETRIES = 5
LIMIT = None

EXCLUDE_KEYS = {
    "score",
    "bm25_rank",
    "dense_rank",
    "bm25_raw_score",
    "dense_raw_score",
    "bm25_rrf_score",
    "dense_rrf_score",
    "has_bm25",
    "has_dense",
}


# =====================================================
# 1. 시스템 프롬프트
# =====================================================

SYSTEM_PROMPT = """
당신은 도서 추천 검색 시스템(RAG 기반 Retrieval System)의 평가자입니다.

입력으로 사용자 검색 의도, RAG 쿼리 정보, 검색된 Top-20 도서 목록이 주어집니다.
당신의 역할은 검색 결과 전체가 사용자 의도와 조건을 얼마나 잘 만족하는지 평가하는 것입니다.

각 항목은 반드시 0점, 1점, 2점 중 하나로 채점하세요.

평가 항목은 다음 5개입니다.

1. result_diversity

- 모든 케이스에서 평가합니다.
- 다양성은 "쿼리 의도를 유지한 상태에서" 장르, 주제, 분위기, 관점, 세부 소재 등이 얼마나 폭넓게 분포하는지를 의미합니다.
- 단순히 무관한 결과가 많이 섞인 경우는 다양성으로 보지 않습니다.

- 2점:
  쿼리 의도를 유지하면서도 최소 3개 이상의 하위 주제/관점/세부 장르가 균형 있게 포함되어 있습니다.

- 1점:
  최소 2개 이상의 하위 주제/관점 차이는 있으나,
  결과가 일부 단일 스타일이나 단일 소재에 편중되어 있습니다.

- 0점:
  대부분이 동일한 소재/스타일/주제에 집중되어 있으며,
  실질적인 다양성이 거의 없습니다.

2. result_focus
- 모든 케이스에서 평가합니다.
- 검색 결과가 특정 장르/주제에 얼마나 집중되어 있는지를 평가합니다.
- 결과 집합의 일관성과 노이즈 비율에 초점을 둡니다.

- 2점:
  검색 결과 대부분이 동일한 핵심 장르/주제 범위 안에 있습니다.
  노이즈가 적고 결과의 방향성이 일관됩니다.

- 1점:
  지정 장르/주제가 포함되어 있으나,
  관련성이 약한 결과나 다른 장르/주제가 함께 섞여 있습니다.

- 0점:
  절반 이상이 지정 장르/주제와 무관합니다.
  결과의 방향성이 크게 분산되어 있습니다.


3. query_alignment
- 모든 케이스에서 평가합니다.
- RAG 쿼리의 핵심 조건(장르, 분위기, 대상 독자, 난이도, 시대 배경, 소재 등)이 실제 검색 결과에 얼마나 반영되었는지를 평가합니다.
- 최종적으로 "사용 가능한 추천 후보군"이 얼마나 확보되었는지를 기준으로 판단합니다.

- 2점:
  최종 추천 후보군이 충분히 확보됨.
  (Top20 중 약 12~20권이 쿼리 의도와 핵심 조건에 적합)
  일부 노이즈는 허용됩니다.

- 1점:
  추천 가능한 책도 존재하지만,
  핵심 조건이 부분적으로만 반영되었거나 노이즈 비율이 높습니다.
  (Top20 중 약 6~11권 적합)

- 0점:
  추천 가능한 책이 거의 없으며,
  검색 결과가 쿼리 의도와 전반적으로 불일치합니다.
  (Top20 중 약 0~5권만 적합)


출력은 반드시 JSON만 반환하세요.
마크다운, 설명 문장, 코드블록은 출력하지 마세요.

출력 형식은 반드시 아래와 같습니다.

{
  "query_id": "",
  "scores": {
    "result_diversity": null,
    "result_focus": null,
    "query_alignment": 0
  },
  "total_score": 0,
  "max_score": 0,
  "pass": false,
  "reason": {
    "result_diversity": "",
    "result_focus": "",
    "query_alignment": ""
  }
}

규칙:
- total_score는 null이 아닌 항목 점수의 합입니다.
- max_score는 null이 아닌 항목 수 × 2입니다.
- pass는 total_score / max_score >= 0.7이면 true입니다.
- 반드시 유효한 JSON만 출력하세요.
"""


# =====================================================
# 2. 유틸 함수
# =====================================================

def make_headers():
    return {
        "Authorization": f"Bearer {CLOVA_API_KEY}",
        "X-NCP-CLOVASTUDIO-REQUEST-ID": str(uuid.uuid4()),
        "Content-Type": "application/json",
        "Accept": "text/event-stream",
    }


def parse_hcx_response(text: str) -> str:
    if "event:" in text:
        last_data = None

        for block in text.split("\n\n"):
            lines = block.strip().splitlines()
            event_type = None
            data_text = None

            for line in lines:
                if line.startswith("event:"):
                    event_type = line.replace("event:", "").strip()
                elif line.startswith("data:"):
                    data_text = line.replace("data:", "").strip()

            if event_type == "result" and data_text:
                last_data = data_text

        if last_data is not None:
            return json.loads(last_data)["message"]["content"]

    try:
        data = json.loads(text)

        if "result" in data:
            return data["result"]["message"]["content"]

        if "message" in data:
            return data["message"]["content"]

    except Exception:
        pass

    raise ValueError(f"응답 파싱 실패:\n{text[:1000]}")


def extract_json_object(llm_text: str) -> dict:
    """
    LLM이 혹시 앞뒤에 텍스트를 붙였을 경우 JSON 부분만 추출.
    """
    try:
        return json.loads(llm_text)
    except Exception:
        pass

    start = llm_text.find("{")
    end = llm_text.rfind("}")

    if start == -1 or end == -1 or start >= end:
        raise ValueError("JSON 객체를 찾지 못했습니다.")

    return json.loads(llm_text[start:end + 1])


def validate_judge_output(obj: dict) -> tuple:
    required_score_keys = [
        "result_diversity",
        "result_focus",
        "query_alignment"
    ]

    if "scores" not in obj:
        return None, "parse_failed", "missing scores"

    scores = obj["scores"]

    for key in required_score_keys:
        if key not in scores:
            return None, "parse_failed", f"missing score key: {key}"

        value = scores[key]

        if value is not None and value not in [0, 1, 2]:
            return None, "parse_failed", f"invalid score {key}: {value}"

    non_null_scores = [
        v for v in scores.values()
        if v is not None
    ]

    total_score = sum(non_null_scores)
    max_score = len(non_null_scores) * 2

    pass_label = False
    if max_score > 0:
        pass_label = (total_score / max_score) >= 0.7

    obj["total_score"] = total_score
    obj["max_score"] = max_score
    obj["pass"] = pass_label

    return obj, "success", None


def safe_str(value, max_len=700):
    if value is None:
        return ""

    text = str(value)

    if len(text) > max_len:
        return text[:max_len] + "..."

    return text


def make_book_for_judge(book: dict) -> dict:
    clean = {}

    for k, v in book.items():
        if k in EXCLUDE_KEYS:
            continue

        if str(k).startswith("retrieval_"):
            continue

        clean[k] = v

    return {
        "rank": clean.get("rank"),
        "isbn": clean.get("isbn"),
        "title": clean.get("title"),
        "author": clean.get("author"),
        "publisher": clean.get("publisher"),
        "publish_date": clean.get("publish_date"),
        "page": clean.get("page"),
        "large_cate": clean.get("large_cate"),
        "mid_cate": clean.get("mid_cate"),
        "small_cate": clean.get("small_cate"),
        "book_intro": safe_str(clean.get("book_intro"), 700),
        "book_index": safe_str(clean.get("book_index") or clean.get("book_index_text"), 700),
        "review": safe_str(clean.get("review_text") or clean.get("review"), 500),
    }


def build_judge_request(query_id: str, books: list) -> dict:
    first = books[0]

    scenario_info = scenario_map.get(query_id, {})
    rag_query = scenario_info.get("rag_query") or first.get("rag_query") or {}

    request_data = {
        "query_id": query_id,

        "query_type": scenario_info.get("query_type"),
        "scenario_type": scenario_info.get("scenario_type"),
        "query_specificity": scenario_info.get("query_specificity"),

        "final_user_input": scenario_info.get("final_user_input"),
        "turns": scenario_info.get("turns"),

        "keyword_query": rag_query.get("keyword_query"),
        "semantic_query": rag_query.get("semantic_query"),
        "filters": rag_query.get("filters"),
        "constraints": rag_query.get("constraints"),
        "score_boost": rag_query.get("score_boost"),
        "anchor": rag_query.get("anchor"),
        "session_signals": rag_query.get("session_signals"),
        "onboarding_signals": rag_query.get("onboarding_signals"),

        "diversity_mode": rag_query.get("diversity_mode"),
        "hard_negative_isbns": hard_negative_map.get(query_id, []),

        "preferred_categories": (
            scenario_info.get("onboarding", {}).get("preferred_categories")
            or rag_query.get("preferred_categories")
            or rag_query.get("onboarding_signals", {}).get("topic")
        ),

        "strategy": first.get("strategy"),
        "retrieval_type": first.get("retrieval_type"),
        "index_name": first.get("index_name"),
        "bm25_weight": first.get("bm25_weight"),
        "dense_weight": first.get("dense_weight"),
    }

    return {
        "request": request_data,
        "retrieved_books": [
            make_book_for_judge(book)
            for book in sorted(books, key=lambda x: x.get("rank", 999))[:TOP_K]
        ],
    }


# =====================================================
# 3. LLM judge 요청 함수
# =====================================================

def blocking_judge_one_query(query_id: str, books: list) -> dict:
    judge_input = build_judge_request(query_id, books)

    body = {
        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": json.dumps(judge_input, ensure_ascii=False, indent=2)
            },
        ],
        "topP": 0.1,
        "topK": 0,
        "max_tokens": 700,
        "temperature": 0.0,
        "repetitionPenalty": 1.0,
        "includeAiFilters": False,
        "seed": 42,
    }

    last_error = None

    for attempt in range(MAX_RETRIES):
        try:
            res = requests.post(
                URL,
                headers=make_headers(),
                json=body,
                timeout=90
            )

            if res.status_code == 200:
                llm_text = parse_hcx_response(res.text)
                obj = extract_json_object(llm_text)
                obj, label_status, parse_error = validate_judge_output(obj)

                if obj is not None:
                    obj["query_id"] = query_id

                return {
                    "query_id": query_id,
                    "judge_status": label_status,
                    "llm_raw_output": llm_text,
                    "llm_error": parse_error,
                    "judge_result": obj,
                    "judge_input": judge_input,
                }

            last_error = f"{res.status_code} / {res.text[:500]}"

            if res.status_code in [429, 500, 502, 503, 504]:
                raise Exception(last_error)

            return {
                "query_id": query_id,
                "judge_status": "api_failed",
                "llm_raw_output": None,
                "llm_error": last_error,
                "judge_result": None,
                "judge_input": judge_input,
            }

        except Exception as e:
            last_error = str(e)

            if attempt == MAX_RETRIES - 1:
                return {
                    "query_id": query_id,
                    "judge_status": "api_failed",
                    "llm_raw_output": None,
                    "llm_error": last_error,
                    "judge_result": None,
                    "judge_input": judge_input,
                }

            wait = min(30, (2 ** attempt) + random.uniform(0, 1))
            time.sleep(wait)


async def judge_one_query_async(query_id, books, semaphore, write_lock, jsonl_file, pbar):
    async with semaphore:
        try:
            result = await asyncio.to_thread(
                blocking_judge_one_query,
                query_id,
                books
            )
        except Exception as e:
            result = {
                "query_id": query_id,
                "judge_status": "api_failed",
                "llm_raw_output": None,
                "llm_error": str(e),
                "judge_result": None,
                "judge_input": None,
            }

    async with write_lock:
        jsonl_file.write(json.dumps(result, ensure_ascii=False) + "\n")
        jsonl_file.flush()

    pbar.update(1)

    return result


# =====================================================
# 4. main
# =====================================================

async def main():
    global scenario_map, hard_negative_map

    with SCENARIO_PATH.open("r", encoding="utf-8") as f:
        scenarios = json.load(f)

    scenario_map = {}

    for s in scenarios:
        qid = s.get("scenario_id") or s.get("query_id")

        final_turn = None
        for turn in s.get("turns", []):
            if turn.get("rag_query"):
                final_turn = turn

        if final_turn is None:
            continue

        scenario_map[qid] = {
            "scenario_id": qid,
            "query_type": s.get("query_type"),
            "scenario_type": s.get("scenario_type"),
            "query_specificity": s.get("query_specificity"),
            "onboarding": s.get("onboarding"),
            "turns": s.get("turns"),
            "final_user_input": final_turn.get("user_input"),
            "rag_query": final_turn.get("rag_query"),
        }

    print("scenario rag_query 수:", len(scenario_map))
    
    with GOLDSET_PATH.open("r", encoding="utf-8") as f:
        goldset = json.load(f)

    hard_negative_map = {}

    for item in goldset:
        qid = item.get("query_id")
        isbn = item.get("isbn")
        final_grade = item.get("final_grade")

        if qid is None or isbn is None:
            continue

        if final_grade == 1:
            hard_negative_map.setdefault(qid, set()).add(str(isbn))

    hard_negative_map = {
        qid: sorted(list(isbns))
        for qid, isbns in hard_negative_map.items()
    }

    print("hard negative 있는 query 수:", len(hard_negative_map))

    with RETRIEVAL_JSON_PATH.open("r", encoding="utf-8") as f:
        candidates = json.load(f)

    grouped = {}

    for book in candidates:
        qid = book.get("query_id")

        if not qid:
            continue

        grouped.setdefault(qid, []).append(book)

    print(f"전체 후보 도서 수: {len(candidates):,}")
    print(f"전체 query 수: {len(grouped):,}")

    processed_query_ids = set()
    already_judged = []

    if JSONL_PATH.exists():
        with JSONL_PATH.open("r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue

                try:
                    item = json.loads(line)
                    qid = item.get("query_id")

                    if qid and qid not in processed_query_ids:
                        processed_query_ids.add(qid)
                        already_judged.append(item)

                except json.JSONDecodeError:
                    continue

        print(f"이미 처리된 query 수: {len(processed_query_ids):,}")

    to_process = [
        (qid, books)
        for qid, books in grouped.items()
        if qid not in processed_query_ids
    ]

    if LIMIT is not None:
        to_process = to_process[:LIMIT]

    print(f"처리 대상 query 수: {len(to_process):,}")

    if not to_process:
        print("처리할 query가 없습니다.")
        return already_judged

    semaphore = asyncio.Semaphore(CONCURRENCY)
    write_lock = asyncio.Lock()
    pbar = tqdm(total=len(to_process), desc="LLM judge 중", unit="query")

    with JSONL_PATH.open("a", encoding="utf-8") as jsonl_file:
        tasks = [
            judge_one_query_async(
                qid,
                books,
                semaphore,
                write_lock,
                jsonl_file,
                pbar
            )
            for qid, books in to_process
        ]

        new_results = await asyncio.gather(*tasks)

    pbar.close()

    all_results = already_judged + list(new_results)

    query_order = {
        qid: i
        for i, qid in enumerate(grouped.keys())
    }

    all_results.sort(
        key=lambda x: query_order.get(x.get("query_id"), 999999)
    )

    with FINAL_JSON_PATH.open("w", encoding="utf-8") as f:
        json.dump(
            all_results,
            f,
            ensure_ascii=False,
            indent=2
        )

    total_success = sum(
        1 for r in all_results
        if r.get("judge_status") == "success"
    )

    total_failed = len(all_results) - total_success

    print("=" * 60)
    print("LLM judge 완료")
    print("총 query 수:", len(all_results))
    print("성공:", total_success)
    print("실패:", total_failed)
    print("JSONL 저장:", JSONL_PATH)
    print("JSON 저장:", FINAL_JSON_PATH)
    print("=" * 60)

    return all_results


results = await main()

scenario rag_query 수: 42
hard negative 있는 query 수: 39
전체 후보 도서 수: 840
전체 query 수: 42
처리 대상 query 수: 42


LLM judge 중:   0%|          | 0/42 [00:00<?, ?query/s]

LLM judge 중: 100%|██████████| 42/42 [02:32<00:00,  3.64s/query]

LLM judge 완료
총 query 수: 42
성공: 42
실패: 0
JSONL 저장: /home/ubuntu/work/dahyeon/evaluation/dataset/final_retriever_llm_judge.jsonl
JSON 저장: /home/ubuntu/work/dahyeon/evaluation/dataset/final_retriever_llm_judge.json


In [48]:
import json
import pandas as pd
from pathlib import Path

# =====================================================
# 파일 경로
# =====================================================

JUDGE_PATH = Path(
    "/home/ubuntu/work/dahyeon/evaluation/dataset/final_retriever_llm_judge.json"
)

# =====================================================
# 로드
# =====================================================

with JUDGE_PATH.open("r", encoding="utf-8") as f:
    judge_results = json.load(f)

print("전체 query 수:", len(judge_results))

# =====================================================
# query별 점수 정리
# =====================================================

rows = []

for item in judge_results:

    query_id = item.get("query_id")
    judge_status = item.get("judge_status")

    judge_result = item.get("judge_result") or {}
    scores = judge_result.get("scores") or {}
    reasons = judge_result.get("reason") or {}

    rows.append({
        "query_id": query_id,
        "judge_status": judge_status,

        "result_diversity": scores.get("result_diversity"),
        "result_focus": scores.get("result_focus"),
        "query_alignment": scores.get("query_alignment"),
        "hard_negative_exclusion": scores.get("hard_negative_exclusion"),
        "category_fit": scores.get("category_fit"),

        "total_score": judge_result.get("total_score"),
        "max_score": judge_result.get("max_score"),
        "pass": judge_result.get("pass"),

        "reason_query_alignment": reasons.get("query_alignment"),
        "reason_category_fit": reasons.get("category_fit"),
    })

judge_df = pd.DataFrame(rows)

# =====================================================
# 점수 높은 순 정렬
# =====================================================

judge_df = judge_df.sort_values(
    ["total_score", "query_id"],
    ascending=[False, True]
)

# =====================================================
# 출력
# =====================================================

pd.set_option("display.max_colwidth", 300)

#display(judge_df)

# =====================================================
# 평균 점수 요약
# =====================================================

judge_df = judge_df.sort_values(
    by="query_id",
    key=lambda x: x.str.extract(r"(\d+)").astype(int)[0]
).reset_index(drop=True)

summary = {
    "query_count": len(judge_df),
    "pass_count": int(judge_df["pass"].sum()),
    "pass_ratio": round(judge_df["pass"].mean(), 4),

    "avg_total_score": round(judge_df["total_score"].mean(), 4),

    "avg_result_diversity":
        round(judge_df["result_diversity"].dropna().mean(), 4),

    "avg_result_focus":
        round(judge_df["result_focus"].dropna().mean(), 4),

    "avg_query_alignment":
        round(judge_df["query_alignment"].dropna().mean(), 4),

    "avg_hard_negative_exclusion":
        round(judge_df["hard_negative_exclusion"].dropna().mean(), 4),

    "avg_category_fit":
        round(judge_df["category_fit"].dropna().mean(), 4),
}

summary_df = pd.DataFrame([summary])

display(judge_df)
display(summary_df)

전체 query 수: 42


,query_id,judge_status,result_diversity,result_focus,query_alignment,hard_negative_exclusion,category_fit,total_score,max_score,pass,reason_query_alignment,reason_category_fit
0,S01,success,2,2,2,None,None,6,6,True,검색된 책들 중 다수가 실존주의적 주제를 다루는 에세이로서 사용자의 요청에 부합합니다.,None
1,S02,success,1,1,0,None,None,2,6,False,"장하준 스타일의 책이 검색되지 않았고, 실무 적용에 특화된 책이 일부에 불과하여 쿼리 의도와 전반적으로 불일치함",None
2,S03,success,2,2,1,None,None,5,6,True,"일부 책들은 이야기 형식으로 쉽게 읽히지만, 모든 책들이 사용자의 '딱딱하지 않은' 스타일 선호에 완전히 부합하지는 않음",None
3,S04,success,1,2,2,None,None,5,6,True,팀장 입문자/중급 독자를 위한 실무 중심 리더십 책들이 잘 반영됨,None
4,S05,success,1,2,2,None,None,5,6,True,번역서가 아닌 한국 작가의 250페이지 이하 인문 에세이 조건에 부합하는 책들이 충분히 확보됨,None
5,S06,success,2,2,2,None,None,6,6,True,"쿼리의 핵심 조건(인간관계, 위로, 공감)에 부합하는 책들이 충분히 포함되어 있음",None
6,S07,success,2,2,2,None,None,6,6,True,"쿼리의 핵심 조건(주식, ETF, 투자 기초, 실전적, 입문 친화적, 2020년 이후, 300페이지 이하)에 부합하는 책이 충분히 포함되어 있음",None
7,S08,success,2,2,2,None,None,6,6,True,"많은 책들이 역사 속 인물들의 삶에 집중한 소설로, 사용자의 요구에 잘 부합합니다. 페이지 제한(400페이지 이하)도 대부분 충족하고 있습니다.",None
8,S09,success,1,1,1,None,None,3,6,False,"일부 책은 요청에 부합하지만, 모든 책이 '공감되면서 힘이 되는' 내용을 보장하지는 않음",None
9,S10,success,2,2,2,None,None,6,6,True,검색된 책들 대부분이 사용자의 쿼리인 '나이 들면서 어떻게 살아야 하나'와 관련된 내용을 다루고 있습니다.,None


,query_count,pass_count,pass_ratio,avg_total_score,avg_result_diversity,avg_result_focus,avg_query_alignment,avg_hard_negative_exclusion,avg_category_fit
0,42,26,0.619,4.8333,1.5238,1.6905,1.619,NaN,NaN


In [24]:
from pathlib import Path
import json
import pandas as pd

# =====================================================
# 파일 경로
# =====================================================

GOLDSET_PATH = Path("/home/ubuntu/work/dahyeon/evaluation/dataset/goldset_final_v2.json")
RETRIEVAL_PATH = Path("/home/ubuntu/work/dahyeon/evaluation/dataset/final_retriever_eval.json")

TOP_K = 20

# =====================================================
# 1. goldset 로드
# =====================================================

with GOLDSET_PATH.open("r", encoding="utf-8") as f:
    goldset = json.load(f)

gold_df = pd.DataFrame(goldset)

# =====================================================
# 2. hard negative map 생성
#    final_grade == 1
# =====================================================

hard_negative_map = {}

for qid, g in gold_df.groupby("query_id"):

    hard_isbns = (
        g[g["final_grade"] == 1]["isbn"]
        .dropna()
        .astype(str)
        .tolist()
    )

    hard_negative_map[qid] = set(hard_isbns)

print("hard negative query 수:", len(hard_negative_map))

# =====================================================
# 3. retrieval 결과 로드
# =====================================================

with RETRIEVAL_PATH.open("r", encoding="utf-8") as f:
    retrieval = json.load(f)

retrieval_df = pd.DataFrame(retrieval)

print("retrieval row 수:", len(retrieval_df))

# =====================================================
# 4. query별 hard negative 포함 여부 계산
# =====================================================

rows = []

for qid, g in retrieval_df.groupby("query_id"):

    hard_negatives = hard_negative_map.get(qid, set())

    if not hard_negatives:
        continue

    g = g.sort_values("rank")

    top20 = (
        g[g["rank"] <= 20]["isbn"]
        .dropna()
        .astype(str)
        .tolist()
    )

    top10 = (
        g[g["rank"] <= 10]["isbn"]
        .dropna()
        .astype(str)
        .tolist()
    )

    rank11_20 = (
        g[(g["rank"] >= 11) & (g["rank"] <= 20)]["isbn"]
        .dropna()
        .astype(str)
        .tolist()
    )

    top20_hard = sorted(list(set(top20) & hard_negatives))
    top10_hard = sorted(list(set(top10) & hard_negatives))
    rank11_20_hard = sorted(list(set(rank11_20) & hard_negatives))

    rows.append({
        "query_id": qid,

        "hard_negative_total": len(hard_negatives),

        "hard_in_top20_count": len(top20_hard),
        "hard_in_top10_count": len(top10_hard),
        "hard_in_11_20_count": len(rank11_20_hard),

        "hard_in_top20": top20_hard,
        "hard_in_top10": top10_hard,
        "hard_in_11_20": rank11_20_hard,
    })

result_df = pd.DataFrame(rows)

# =====================================================
# 5. 전체 통계
# =====================================================

print("=" * 60)

print("Top20 안에 hard negative 있는 query 수:")
print((result_df["hard_in_top20_count"] > 0).sum())

print()

print("Top10 안에 hard negative 있는 query 수:")
print((result_df["hard_in_top10_count"] > 0).sum())

print()

print("11~20위에만 hard negative 있는 query 수:")
print(
    (
        (result_df["hard_in_top10_count"] == 0)
        &
        (result_df["hard_in_11_20_count"] > 0)
    ).sum()
)

print("=" * 60)

# =====================================================
# 6. 상세 확인
# =====================================================

display(
    result_df.sort_values(
        ["query_id", "hard_in_top10_count", "hard_in_11_20_count"],
        ascending=[True, False, False],
        key=lambda col: (
            col.str.extract(r"(\d+)").astype(int)[0]
            if col.name == "query_id"
            else col
        )
    ).reset_index(drop=True)
)

hard negative query 수: 39
retrieval row 수: 840
Top20 안에 hard negative 있는 query 수:
38

Top10 안에 hard negative 있는 query 수:
36

11~20위에만 hard negative 있는 query 수:
2


,query_id,hard_negative_total,hard_in_top20_count,hard_in_top10_count,hard_in_11_20_count,hard_in_top20,hard_in_top10,hard_in_11_20
0,S01,69,15,6,9,"[9791138801768, 9791139201734, 9791141061876, 9791141903374, 9791141996772, 9791158741891, 9791164050840, 9791193916537, 9791194006435, 9791194299202, 9791194648079, 9791196797720, 9791196956905, 9791198808516, 9791198954527]","[9791139201734, 9791141996772, 9791193916537, 9791194299202, 9791198808516, 9791198954527]","[9791138801768, 9791141061876, 9791141903374, 9791158741891, 9791164050840, 9791194006435, 9791194648079, 9791196797720, 9791196956905]"
1,S03,43,6,3,3,"[9788936812300, 9788958077275, 9791157846313, 9791164710348, 9791168623262, 9791193778036]","[9791157846313, 9791164710348, 9791168623262]","[9788936812300, 9788958077275, 9791193778036]"
2,S04,50,4,0,4,"[9788959890316, 9788972574699, 9788993111071, 9788996464648]",[],"[9788959890316, 9788972574699, 9788993111071, 9788996464648]"
3,S05,45,6,2,4,"[9788972676027, 9788997483495, 9791162757017, 9791189930295, 9791191013917, 9791198898845]","[9788972676027, 9791191013917]","[9788997483495, 9791162757017, 9791189930295, 9791198898845]"
4,S06,23,3,3,0,"[9788975070648, 9791141037857, 9791193341513]","[9788975070648, 9791141037857, 9791193341513]",[]
5,S07,39,4,1,3,"[9788947500746, 9788998342784, 9791160023527, 9791170432821]",[9788947500746],"[9788998342784, 9791160023527, 9791170432821]"
6,S08,50,8,3,5,"[9788955520026, 9788972531296, 9788984989085, 9788988295106, 9788989721321, 9788992673082, 9788993192001, 9788993379143]","[9788972531296, 9788993192001, 9788993379143]","[9788955520026, 9788984989085, 9788988295106, 9788989721321, 9788992673082]"
7,S09,48,13,8,5,"[9788932006673, 9788936436889, 9788936438005, 9788936616083, 9788952770394, 9788954637800, 9788960781672, 9788979197570, 9788995655245, 9791137296541, 9791156290230, 9791189217358, 9791190492829]","[9788932006673, 9788936438005, 9788936616083, 9788954637800, 9791137296541, 9791156290230, 9791189217358, 9791190492829]","[9788936436889, 9788952770394, 9788960781672, 9788979197570, 9788995655245]"
8,S10,21,2,0,2,"[9788964200452, 9791190162524]",[],"[9788964200452, 9791190162524]"
9,S11,61,14,7,7,"[9788932006673, 9788936616083, 9788937488962, 9788941329251, 9788954611275, 9788980502042, 9788984981782, 9788990707635, 9791138833530, 9791141966881, 9791156290230, 9791191797169, 9791192265599, 9791194595731]","[9788932006673, 9788936616083, 9788980502042, 9788984981782, 9791138833530, 9791141966881, 9791156290230]","[9788937488962, 9788941329251, 9788954611275, 9788990707635, 9791191797169, 9791192265599, 9791194595731]"


# Human 정성적평가

In [52]:
from pathlib import Path
import json
import pandas as pd

RETRIEVAL_PATH = Path("/home/ubuntu/work/dahyeon/evaluation/dataset/final_retriever_eval.json")

with RETRIEVAL_PATH.open("r", encoding="utf-8") as f:
    retrieval = json.load(f)

df = pd.DataFrame(retrieval)

def short_text(x, n=180):
    if x is None:
        return ""
    return str(x).replace("\n", " ")[:n]

def show_query_results(qid):
    qdf = (
        df[df["query_id"] == qid]
        .sort_values("rank")
        .head(20)
        .copy()
    )

    if qdf.empty:
        print(f"{qid} 결과 없음")
        return

    qdf["book_intro_short"] = qdf["book_intro"].apply(lambda x: short_text(x, 180))

    view_cols = [
        "rank",
        "isbn",
        "title",
        "author",
        "large_cate",
        "mid_cate",
        "small_cate",
        "page",
        "publish_date",
        "book_intro_short",
    ]

    print("=" * 100)
    print(f"QUERY: {qid}")
    print("=" * 100)

    print("\n[large_cate 분포]")
    display(qdf["large_cate"].explode().value_counts().reset_index())

    print("\n[mid_cate 분포]")
    display(qdf["mid_cate"].explode().value_counts().reset_index())

    print("\n[Top-20 결과]")
    display(qdf[view_cols].reset_index(drop=True))


# 예시
show_query_results("S36")

QUERY: S36

[large_cate 분포]


,large_cate,count
0,소설,20



[mid_cate 분포]


,mid_cate,count
0,일본소설,12
1,라이트노벨,4
2,장르소설,4
3,중국소설,2
4,한국소설,2
5,프랑스소설,1



[Top-20 결과]


,rank,isbn,title,author,large_cate,mid_cate,small_cate,page,publish_date,book_intro_short
0,1,9791138031349,옆집 천사님 때문에 어느샌가 인간적으로 타락한 사연 8,사에키상 저/하네코토 그림/JYH 역,[소설],"[라이트노벨, 일본소설]","[라이트노벨, 일본라이트노벨]",280,2023-08-08,"“오늘은…… 집에 가지 않아도, 될까요……?” 부산했던 문화제도 끝나고, 일상이 돌아온다. 아마네는 마히루에게 자기 마음을 다시 말로 전하고, 함께 장래를 맹세했다. 나아가 그것을 형태로 갖춘 선물을 주고 싶어져 익숙하지 않은 아르바이트를 시작하는 아마네. 한편, 아마네의 귀가를 기다리게 된 마히루는 쓸쓸함을 느끼면서도"
1,2,9788959527472,밤하늘은 올려다보는 그대에게 상냥하게,마쿠라기 미루타 저/손지상 역,[소설],[일본소설],[일본소설일반],288,2019-01-25,"“달이 참 예쁘네요” 전격소설대상에서 발탁된 젊은 신인작가의 야심작! 나쓰메 소세키의 사랑의 언어로 가슴을 울리는 밤하늘의 기적! 시부야의 밤을 밝히는 애드벌룬. 기구에 매달린 현수막에는, 밤거리를 즐기는 사람들의 SNS글이 차례대로 흘러간다. 애드벌룬 관리인 아르바이트를 하고 있던 요코모리는 어느 날 끊임없이 흐르는 S"
2,3,9791127877989,일주일에 한 번 클래스메이트를 사는 이야기 2,하네다 우사 저/U35 그림/이소정 역,[소설],"[라이트노벨, 일본소설]","[라이트노벨, 일본라이트노벨]",334,2024-10-10,"긴 방학은 우울하다. 혼자 있는 것은 익숙하지만, 혼자 있는 것을 좋아하진 않는다. 고등학교 마지막 여름 방학은 모두 바쁘다. 센다이와 만나는 것 역시 「방과 후」뿐이라고 정했다. 그런데 그녀는 『과외를 해주겠다』라고 제안해왔다. ……키스한 걸 신경 쓰는 건 나뿐이야? 재미없어. 시시해. 곧 다가올 긴 방학을 앞두고, 키"
3,4,9791191560015,잔잔한 파도에 빠지다 = なぎに溺れる,아오바 유 저/김지영 역,[소설],[],[],360,2021-05-26,"나는 뭐든 할 수 있다, 무엇이든 될 수 있다, 어디로든 갈 수 있다… 그런 믿음이 사라진 건 언제부터였을까. 일도 연애도 타성에 젖은 채 하루하루를 보내던 어느 날, 우연히 유튜브에서 화제가 된 무명 아티스트의 곡을 발견한다. 그 곡에 엄청난 끌림을 느끼지만, 며칠 뒤 그 아티스트의 공식 사이트에서 ‘2018년 10월"
4,5,9788956601526,비 2 = 天瓢,차오원셴 저,[소설],[중국소설],[중국소설일반],320,2007-07-01,차오원쉬엔의 신작 장편 <비>는 그동안 그의 작품에서 보아왔던 소년들의 성장을 넘어 처음으로 선사하는 러브스토리로 그의 전작들과 마찬가지로 서정적이고 아름다운 묘사가 소설 전체에 가득하다. 하루아침에 모든 것이 변해버린 시대의 폭풍에 휩쓸린 한 여자와 그녀를 사랑하지만 가슴 깊이 묻은 말조차 제대로 건네지 못하는 남자 그
5,6,9788925505510,하루가 떠나면,아스카이 치사 저,[소설],[일본소설],[일본소설일반],296,2007-02-26,"감각적인 문체로 일본 독자를 사로잡은 일본문단의 기대주, 아스카이 치사의 제18회 소설스바루 신인문학상 수상작. 이혼 가정이라는 평범치 않은 가정환경 속에서 살아가는 젊은이들의 관대함과 타인에 대한 이해를 산뜻하게 그려냈다. 하루는 남매가 어릴 적 공원에서 주워와 끝까지 책임지겠다고 14년을 길러온 개의 이름이다. 남매에"
6,7,9788956601519,비 1 = 天瓢,차오원셴 저,[소설],[중국소설],[중국소설일반],312,2007-07-01,차오원쉬엔의 신작 장편 <비>는 그동안 그의 작품에서 보아왔던 소년들의 성장을 넘어 처음으로 선사하는 러브스토리로 그의 전작들과 마찬가지로 서정적이고 아름다운 묘사가 소설 전체에 가득하다. 하루아침에 모든 것이 변해버린 시대의 폭풍에 휩쓸린 한 여자와 그녀를 사랑하지만 가슴 깊이 묻은 말조차 제대로 건네지 못하는 남자 그
7,8,9788957590065,첫날밤,하야시 마리코 저/양윤옥 역,[소설],[일본소설],[일본소설일반],292,2003-12-15,"독특한 시선으로 씌여진 11편의 단편집. 혼기를 놓친 외동딸이 자궁 제거 수술을 받기 전날밤 그 아버지의 심정이나 혹은 어릴 적 친하긴 했으나 여전히 질투가 남아있는 친구의 딸을 강에서 만나자 이 아이를 강물에 띄워 보내면 어떨까 하고 생각이 드는 단편 등, 어찌보면 괴기스럽고 낯선 일상에 대한 시선을 만나볼 수 있다."
8,9,9791161308098,고요한 연못에 내린 비,원주희 저,[소설],[],[],416,2017-05-10,"처음 만나던 날 기억하세요? 손을 뻗으면 푸른 물이 들 것 같은 하늘에서 꽃비가 우수수 내렸어요. 사방에서 향긋한 흙냄새와 꽃 냄새가 나고 계곡물 소리와 산새 소리가 마치 노래 같았어요. 아직도 귀에 생생해요. 빨리 오라며 채희가 재촉하는 소리, 눈이가 왕왕 짓는 소리, 계곡물에 한섭이가 던진 조약돌이 통통 튀는 소리,"
9,10,9788989675723,태양의 노래 - 태양이 지면 만나러 갈게,카와이 나츠키 저 / 김영주 역,[소설],"[일본소설, 장르소설]","[드라마/영화소설, 일본소설일반]",263,2007-02-22,"영화 '태양의 노래'를 소설로 다시 쓴 작품. 색소성건피증이라는 희귀병을 앓는 소녀의 이야기. 작은 희망이라도 부여잡으려는 그녀에게 점점 몰려드는 전신마비와 죽음에 대한 두려움, 그 틈바구니에서 자신만의 음악을 만들고, 짝사랑을 현실의 사랑으로 일궈내고픈 그녀의 몸부림은 독자와 관객의 눈물샘을 터뜨렸다. 이 책에서는 영화"
